# Two frozen submission candidates

GPU T4 ×2, Internet Off. Подключите исходный base и новый runtime.zip (Kaggle распаковывает его сам). Обновлённый runtime содержит train-only checkpoint и исторические SigLIP scores. Один запуск создаёт 056_compact.csv и 057_quality.csv в /kaggle/working. Это инференс готовых моделей, не полный fresh train→test benchmark.


In [ ]:
SELECTED = ["056_compact", "057_quality"]
SEEDS = None  # For training configs only.


In [ ]:
from pathlib import Path
import hashlib, json, os, sys, zipfile
import torch

assert torch.cuda.device_count() == 2, "Select GPU T4 x2"
input_root = Path(os.environ.get("KAGGLE_INPUT_ROOT", "/kaggle/input"))
work_root = Path(os.environ.get("KAGGLE_WORK_ROOT", "/kaggle/working"))
projects = sorted({p.parent for p in input_root.rglob("package.json")
                   if all((p.parent / name).is_dir() for name in ("src", "configs", "inputs"))})
assert len(projects) <= 1, f"Several unpacked project datasets found: {projects}"
if projects:
    # Kaggle exposes uploaded ZIP contents as a read-only dataset directory.
    # Code, configurations and inputs can be read directly from that directory.
    project = projects[0]
else:
    archives = list(input_root.rglob("base.zip"))
    assert len(archives) == 1, "Attach one project dataset with package.json, src, configs, inputs or base.zip"
    project = work_root / "project"
    project.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as bundle:
        for entry in bundle.infolist():
            dest = (project / entry.filename).resolve()
            assert dest.is_relative_to(project.resolve()), "Unsafe archive path"
        bundle.extractall(project)

manifest = json.loads((project / "package.json").read_text(encoding="utf-8"))
assert manifest.get("version") == 1, "Unsupported package manifest version"
for name, digest in manifest["files"].items():
    path = (project / name).resolve()
    assert path.is_relative_to(project.resolve()), "Unsafe manifest path"
    with path.open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == digest, f"Package checksum mismatch: {name}"
inputs = project / "inputs"
foundations = project / "foundations" if (project / "foundations").is_dir() else None
# Runtime overlay; optional train-only foundation weights remain read-only in its input dataset.
overlays = sorted(input_root.rglob("runtime.json"))
assert len(overlays) <= 1, f"Several runtime overlays found: {overlays}"
if overlays:
    overlay = json.loads(overlays[0].read_text(encoding="utf-8"))
    assert overlay.get("version") == 1, "Unsupported runtime overlay"
    runtime_project = work_root / "runtime_project"
    runtime_project.mkdir(parents=True, exist_ok=True)
    for name, digest in overlay["files"].items():
        source = (overlays[0].parent / name).resolve()
        dest = (runtime_project / name).resolve()
        assert source.is_relative_to(overlays[0].parent.resolve()), "Unsafe overlay source"
        assert dest.is_relative_to(runtime_project.resolve()), "Unsafe overlay target"
        assert name.startswith(("src/", "configs/", "foundations/")), "Unexpected overlay content"
        if name.startswith("foundations/"):
            with source.open("rb") as stream:
                assert hashlib.file_digest(stream, "sha256").hexdigest() == digest, f"Foundation checksum mismatch: {name}"
            continue  # Read frozen weights directly from input; do not copy them into output.
        content = source.read_bytes()
        assert hashlib.sha256(content).hexdigest() == digest, f"Runtime checksum mismatch: {name}"
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(content)
    project = runtime_project
    if (overlays[0].parent / "foundations").is_dir():
        foundations = overlays[0].parent / "foundations"
sys.path.insert(0, str(project))
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
roots = sorted({p.parent for p in input_root.rglob("train.csv")
                if (p.parent / "val.csv").is_file()
                and (p.parent / "train").is_dir() and (p.parent / "val").is_dir()})
assert len(roots) == 1, f"Competition CSVs and train/val image directories must share one root: {roots}"
data = roots[0]
print("Project:", project, "Data:", data, "Foundations:", foundations)


Два кандидата используют общие CNN/TTA/union/counter вычисления. Сначала test и два CSV, затем диагностическая val-оценка. Никакого обучения на test/val. При возвращении к исследованиям замените SELECTED прежними count/union/resnet/siglip конфигурациями. Выходные артефакты используются только для записи; кэш находится в /kaggle/temp/foundation.


In [ ]:
from src import runtime
assert hasattr(runtime, "zero_loss"), "Update the runtime dataset and restart the kernel"
from src.config import load_config
families = [load_config(project / "configs" / (name + ".yaml"))["family"] for name in SELECTED]
if "submission" in families:
    assert set(families) == {"submission"}, "Run submissions separately from training configs"
    from src.submit import run
    results = run(project=project, data=data, inputs=inputs, foundations=foundations,
                  selected=SELECTED, splits=("test", "val"),
                  work=Path("/kaggle/temp/foundation"),
                  output=Path("/kaggle/working/artefacts/results/submissions"),
                  submission_dir=Path("/kaggle/working"))
    print("CSV: /kaggle/working/056_compact.csv and /kaggle/working/057_quality.csv")
else:
    from src.suite import run_suite
    results = run_suite(project=project, data=data, inputs=inputs,
                        selected=SELECTED, seeds=SEEDS, verified_inputs=True, foundations=foundations,
                        work=Path("/kaggle/temp/foundation"), output=Path("/kaggle/working/artefacts"))
results
